## Situación de Knapsack 0/1

### Enunciado

El grupo *Raccon* es un equipo de arquitectura de software dedicado a desarrollar el sistema de software *Raccon Analytics*, un analizador de tendencias de redes sociales en el cuál el usuario final puede hacer búsquedas tentativas para youtube o google trends, y el sistema devuelve búsquedas relacionadas y keywords inferidas con inteligencia artificial, junto con contenido reciente y viral de dichas redes sociales, para que el usuario tenga información valiosa para poder subir contenido público en las redes y viralizarlo.

Este grupo actualmente tiene sus componentes del sistema desplegados de manera distribuida en 4 máquinas de cómputo diferentes; cómo este grupo es muy profesional y se preocupa por la calidad del software, están tomando decisiones arquitectónicas para que su sistema cumpla con el atributo de calidad de confiabilidad, que consiste en garantizar una buena disponibilidad, resiliencia y tolerancia a fallos.
El mes que viene (Julio de 2026) comienza uno de los eventos más grandes de internet organizado por uno de los creadores de contenido más grandes, se aproxima la velada del año en su sexta edición organizada por Ibai Llanos, y con sigo muchos creadores más pequeños verán oportuno obtener visitas gracias a ello, y usarán la aplicación *Raccon Analytics* para evaluar tendencias y subir el contenido correcto.

Actualmente el sistema desplegado en las 4 computadoras soporta un estimado de 300 usuarios concurrentes, pero debido al incremento de usuarios que generará la velada, el equipo *Raccon* deberá garantizar un correcto funcionamiento con 1000 usuarios concurrentes, por lo que optan por contratar temporalmente poder computacional en los servicios de AWS, y desplegar allí réplicas de los componentes de mayor impacto en el negocio junto con balanceadores de carga y un service discovery.

El poder de cómputo a nivel de procesamiento que ofrece el contrato es más que suficiente para el equipo *Raccon* ya que el sistema mayormente solo maneja el flujo de datos y peticiones junto con API's externas, pero donde sí hay un limitante importante es en la memoria RAM.
Dicho contrato ofrece un cluster con un total de 8192MB de RAM, y dado que el equipo desplegará docker swarm junto con sus balanceadores y su DNS (Service Discovery), la memoria que queda para componentes del sistema se software son 8000MB.

Está claro que desplegar la totalidad de componentes del sistema en AWS no es viable, ya que con 1000 usuarios concurrentes, simplemente la RAM no daría abasto, por lo que por medio de pruebas de rendimiento desarrolladas por el equipo, por cada componente individual se obtuvo la cantidad de RAM en megabytes (MB) necesaria para ejecutarlo, y los resultados los anotaron en un listado de pesos denominado $W$.
Adicionalmente, hubo una discusión por parte del equipo sobre el nivel de impacto en el negocio que generaba cada uno de los componentes del sistema, y a cada uno le asignaron un número entero positivo de manera tal que un número alto indicaba alto nivel de impacto e importancia, mientras que un número bajo indicaba bajo nivel de impacto e importancia, y estos valores de impacto los anotaron en un listado de valores denominado $V$.

El equipo quiere maximizar la sumatoria de niveles de impacto de los componentes desplegados en AWS, y los componentes que no se desplieguen en AWS se mantendrán desplegados en las máquinas del equipo.

### Modelamiento matemático del problema

Definimos las listas de pesos y de valores:

$W = \{601, 7902, 6, 4520, 112, 7890, 334, 1200, 5600, 45, 6780, 230, 4400, 900, 3100, 7200, 50, 88, 3900, 7500, 2100, 6400, 500, 4200, 7700, 150, 3300, 120, 5900, 6800, 2500, 400, 7100, 950, 4800, 10, 6200, 3500, 800, 5500, 2700, 600, 7300, 1800, 4900, 7600, 320, 5400, 2200, 110\}$

$V = \{100, 567, 802, 345, 999, 120, 450, 670, 230, 890, 410, 560, 320, 770, 150, 600, 950, 430, 210, 500, 820, 390, 700, 480, 110, 660, 370, 920, 280, 530, 740, 400, 850, 190, 620, 980, 300, 550, 790, 440, 250, 690, 360, 810, 460, 130, 720, 260, 510, 930\}$

$|W| = |V| = 50$

$W_i$ es el costo en RAM del i-ésimo item

$V_i$ es el impacto estimado del i-ésimo item

#### Variables de decisión

Hay 50 variables binarias, $X_i$ es una variable binaria que indica si se elije o no el i-ésimo item

$0 <= i < |V|$

#### Función objetivo

$Max$ $z =$ $\sum_{i=0}^{49}$ $V_i$ $\cdot$ $X_i$

#### Restricciones

Restricción binaria: $X_i \in \{0, 1\}$ $\forall$ $i \in \mathbb{Z} \cap [0, 49]$

Restricción de máximo uso de RAM: $\sum_{i=0}^{49}$ $W_i$ $\cdot$ $X_i$ $\leq$ $8000$

#### Solución con programación dinámica

In [1]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <cassert>
#include <cstring>
using namespace std;

In [2]:
const int mxW = 8000;

In [3]:
vector<bool> Knapsack(vector<int>& weig, vector<int>& val) {
  int N = weig.size();
  int dp[N][mxW + 1];
  bool take[N][mxW + 1];
  memset(dp, 0, sizeof(dp));
  memset(take, 0, sizeof(take));

  dp[0][weig[0]] = val[0];
  take[0][weig[0]] = true;

  for (int phase = 1; phase < N; ++phase) {
    int idx = phase;
    for (int w = 0; w <= mxW; ++w) {
      // Dont take
      dp[idx][w] = dp[idx - 1][w];

      // Take
      if (weig[idx] <= w && dp[idx - 1][w - weig[idx]] + val[idx] >= dp[idx][w]) {
        dp[idx][w] = dp[idx - 1][w - weig[idx]] + val[idx];
        take[idx][w] = true;
      }
    }
  }
  int best_val = *max_element(dp[N - 1], dp[N - 1] + mxW + 1);

  // Build the solution
  int current_weig = -1;
  for (int w = 0; w <= mxW; ++w) {
    if (best_val == dp[N - 1][w]) {
      current_weig = w;
      break;
    }
  }
  assert(current_weig > -1);
  int check_val = 0;
  vector<bool> X(N, false);
  for (int i = N - 1; i >= 0; --i) {
    if (take[i][current_weig]) {
      X[i] = true;
      current_weig -= weig[i];
      check_val += val[i];
    }
  }
  assert(current_weig == 0);
  assert(best_val == check_val);
  return X;
}

In [4]:
vector<int> W = {601, 7902, 6, 4520, 112, 7890, 334, 1200, 5600, 45, 6780, 230, 4400, 900, 3100, 7200, 50, 88, 3900, 7500, 2100, 6400, 500, 4200, 7700, 150, 3300, 120, 5900, 6800, 2500, 400, 7100, 950, 4800, 10, 6200, 3500, 800, 5500, 2700, 600, 7300, 1800, 4900, 7600, 320, 5400, 2200, 110};
vector<int> V = {100, 567, 802, 345, 999, 120, 450, 670, 230, 890, 410, 560, 320, 770, 150, 600, 950, 430, 210, 500, 820, 390, 700, 480, 110, 660, 370, 920, 280, 530, 740, 400, 850, 190, 620, 980, 300, 550, 790, 440, 250, 690, 360, 810, 460, 130, 720, 260, 510, 930};
vector<bool> mask = Knapsack(W, V);

int best_val = 0;
int best_weig = 0;

cout << "Los componentes a desplegar en AWS son: " << endl;
for (int i = 0; i < mask.size(); ++i) {
  if (mask[i]) {
    cout << i << ",\n"[i + 1 == mask.size()] << " \n"[i + 1 == mask.size()];
    best_val += V[i];
    best_weig += W[i];
  }
}

cout << "Mejor valor de impacto lograble con 8000MB = " << best_val << " puntos" << endl;
cout << "Memoria RAM usada = " << best_weig << "MB" << endl;

Los componentes a desplegar en AWS son: 
2, 4, 6, 7, 9, 11, 13, 16, 17, 22, 25, 27, 31, 35, 38, 41, 43, 46, 49

Mejor valor de impacto lograble con 8000MB = 14121 puntos
Memoria RAM usada = 7775MB


#### Solución óptima del equipo

Después de solucionar el problema usando programación dinámica observamos que el movimiento más inteligente es que de los 50 componentes del sistema que tiene el equipo, se desplieguen en AWS los componentes 2, 4, 6, 7, 9, 11, 13, 16, 17, 22, 25, 27, 31, 35, 38, 41, 43, 46 y 49, teniendo en cuenta que los componentes están enumerados desde 0.
Esto proporciona al equipo un valor de impacto total de 14121 puntos usando un total de 7775MB de RAM por parte de estos componentes desplegados.

## Situación de Set Covering

### Enunciado